In [ ]:
# ================================================================
# andre_becker_PB_TP4 — K-Means como Engenharia de Features + SVM/RF
# Disciplina: Projeto de Bloco — TP4 | Aluno: André Luis Becker | Data: 2025-09-01
# Ambiente: Google Colab — **ATENÇÃO**: Este notebook contém APENAS UMA seção de código.
# Cada questão está identificada via comentários (# Q1, # Q2, ...). Não existem células de texto.
# ================================================================

# ================================
# Imports e Configurações Iniciais
# ================================
import os
import sys
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import List, Tuple, Dict, Optional

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_score, roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, RocCurveDisplay
)
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

# =====================================================================
# Utilitários gerais — Funções modulares (Clean Code) para reuso no TP4
# =====================================================================


# =====================
# Carregar e preparar o dataset Spotify (simplificado)
# =====================
import kagglehub
from kagglehub.dataset_adapter import KaggleDatasetAdapter

def carregar_dataset_spotify(file_path: str = "dataset.csv", n_amostra: int = 30000, seed: int = 42):
    """Carrega o Spotify Tracks Dataset direto do Kaggle e retorna um DataFrame amostrado."""
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "maharshipandya/-spotify-tracks-dataset",
        file_path
    )
    df = df.sample(n=n_amostra, random_state=seed).reset_index(drop=True)
    print(f"[OK] Dataset carregado com {len(df)} linhas (amostra de {n_amostra}).")
    return df

def preparar_dados(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """Seleciona apenas colunas numéricas, cria rótulo binário (popularidade),
    remove colunas inválidas, e retorna X (features) e y (target binário).
    Regra do alvo: 'is_popular' = popularidade >= mediana (balanceado).
    Evita vazamento de informação removendo 'popularity' das features.
    """
    # Seleciona apenas numéricos
    df_num = df.select_dtypes(include=[np.number]).copy()

    # Verificações básicas
    if 'popularity' not in df_num.columns:
        raise ValueError("A coluna 'popularity' é obrigatória para criar o alvo.")

    # Cria alvo binário a partir da mediana (classes mais balanceadas)
    mediana = df_num['popularity'].dropna().median()
    y = (df_num['popularity'] >= mediana).astype(int)

    # Remove a coluna de popularidade das features (para não vazar o rótulo)
    X = df_num.drop(columns=['popularity'])

    # Remove colunas com cardinalidade 1 (constantes) que não agregam
    nunique = X.nunique(dropna=False)
    cols_const = nunique[nunique <= 1].index.tolist()
    if cols_const:
        X = X.drop(columns=cols_const)

    # Imputação simples para missings (alguns datasets têm NaNs)
    X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=X.columns)

    # (Opcional) Normalização posterior via Pipeline para cada modelo
    return X, y


def split_train_test(X: pd.DataFrame, y: pd.Series, test_size: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Divide em treino/teste estratificado."""
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)


def avaliar_kmeans_treino(X_train: pd.DataFrame, ks: List[int], amostra_sil: int = 10000) -> pd.DataFrame:
    """Roda K-Means para diferentes k (apenas no conjunto de treino),
    computa inércia e silhouette (em amostra) e retorna DataFrame com métricas.
    """
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)

    n = Xs.shape[0]
    amostra = min(amostra_sil, n)
    idx = np.random.choice(n, size=amostra, replace=False)

    resultados = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init='auto', random_state=RANDOM_STATE)
        labels = km.fit_predict(Xs)
        inertia = km.inertia_

        # Silhouette em amostra (labels e features da amostra)
        sil = silhouette_score(Xs[idx], labels[idx]) if k > 1 and amostra >= 100 else np.nan

        resultados.append({"k": k, "inertia": inertia, "silhouette": sil})

    return pd.DataFrame(resultados)


def escolher_k(df_metrics: pd.DataFrame) -> int:
    """Escolhe k com base em (1) melhor silhouette; (2) critério de 'cotovelo' simplificado.
    Se silhouette tiver múltiplos empates ou NaN, usa o menor k com bom ganho de inércia.
    """
    dfm = df_metrics.dropna(subset=['silhouette']).copy()
    if not dfm.empty:
        k_sil = int(dfm.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]['k'])
        return k_sil

    # Fallback: cotovelo (maior queda relativa de inércia)
    inertia = df_metrics['inertia'].values.astype(float)
    ks = df_metrics['k'].values.astype(int)

    # Queda relativa entre pontos consecutivos
    quedas = (inertia[:-1] - inertia[1:]) / np.maximum(inertia[:-1], 1e-9)
    if len(quedas) == 0:
        return int(ks[0])
    idx_best = int(np.argmax(quedas))
    return int(ks[idx_best + 1])


def treinar_kmeans_e_distancia(X_train: pd.DataFrame, X_test: pd.DataFrame, k: int) -> Tuple[pd.DataFrame, pd.DataFrame, KMeans, StandardScaler]:
    """Treina K-Means no treino (features padronizadas) e adiciona 'dist_to_centroid'
    tanto ao treino quanto ao teste. Retorna os DataFrames expandidos + objetos.
    """
    scaler = StandardScaler()
    Xs_train = scaler.fit_transform(X_train)
    Xs_test  = scaler.transform(X_test)

    km = KMeans(n_clusters=k, n_init='auto', random_state=RANDOM_STATE)
    km.fit(Xs_train)

    # Distância ao centróide mais próximo (no espaço padronizado)
    dist_train = pairwise_distances(Xs_train, km.cluster_centers_).min(axis=1)
    dist_test  = pairwise_distances(Xs_test,  km.cluster_centers_).min(axis=1)

    X_train_exp = X_train.copy()
    X_test_exp  = X_test.copy()
    X_train_exp['dist_to_centroid'] = dist_train
    X_test_exp['dist_to_centroid']  = dist_test

    return X_train_exp, X_test_exp, km, scaler


def montar_pipelines_e_grids(modelo: str) -> Tuple[Pipeline, Dict[str, List]]:
    """Cria Pipeline (imputação + padronização + modelo) e grade de hiperparâmetros."""
    if modelo == 'svm_linear':
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel='linear', probability=True, random_state=RANDOM_STATE))
        ])
        grid = {
            "clf__C": [0.1, 1, 10, 100]
        }
    elif modelo == 'svm_poly':
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel='poly', probability=True, random_state=RANDOM_STATE))
        ])
        grid = {
            "clf__C": [0.1, 1, 10],
            "clf__degree": [2, 3],
            "clf__gamma": ['scale', 'auto'],
            "clf__coef0": [0.0, 1.0]
        }
    elif modelo == 'svm_rbf':
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))
        ])
        grid = {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ['scale', 'auto']
        }
    elif modelo == 'rf':
        pipe = Pipeline([
            # RF não precisa de scaler, mas manter padrão é ok se escolhermos None
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
        ])
        grid = {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_leaf": [1, 5],
            "clf__max_features": ['sqrt', 0.5]
        }
    else:
        raise ValueError("Modelo inválido.")
    return pipe, grid


def treinar_avaliar_grid(modelo: str, X_train: pd.DataFrame, y_train: pd.Series,
                         X_test: pd.DataFrame, y_test: pd.Series,
                         cv_splits: int = 5) -> Tuple[GridSearchCV, Dict[str, float]]:
    """Executa GridSearchCV e avalia em teste. Retorna o grid e as métricas."""
    pipe, grid = montar_pipelines_e_grids(modelo)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(pipe, grid, scoring='f1', cv=cv, n_jobs=-1, refit=True, verbose=0)
    gs.fit(X_train, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test)
    if hasattr(best.named_steps['clf'], "predict_proba") and callable(getattr(best.named_steps['clf'], 'predict_proba', None)):
        y_proba = best.predict_proba(X_test)[:, 1]
    else:
        # fallback para decisão binária aproximada
        try:
            y_score = best.decision_function(X_test)
            # normaliza para 0-1 para AUC consistente
            y_proba = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)
        except Exception:
            y_proba = y_pred.astype(float)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) == 2 else np.nan
    }
    return gs, metrics


def comparar_modelos(feature_set_nome: str,
                     X_train: pd.DataFrame, y_train: pd.Series,
                     X_test: pd.DataFrame, y_test: pd.Series) -> Tuple[pd.DataFrame, Dict[str, GridSearchCV]]:
    """Treina SVM (linear, poly, rbf) e RandomForest com GridSearch.
    Retorna tabela de métricas e dicionário com grids para uso posterior.
    """
    resultados = []
    grids = {}

    for modelo in ['svm_linear', 'svm_poly', 'svm_rbf', 'rf']:
        print(f"\\n[GridSearch] {feature_set_nome} — {modelo}...")
        gs, m = treinar_avaliar_grid(modelo, X_train, y_train, X_test, y_test, cv_splits=5)
        grids[modelo] = gs

        linha = {
            "feature_set": feature_set_nome,
            "modelo": modelo,
            **m,
            "best_params": gs.best_params_
        }
        resultados.append(linha)

    df_res = pd.DataFrame(resultados)
    cols_ordem = ["feature_set", "modelo", "accuracy", "precision", "recall", "f1", "auc", "best_params"]
    df_res = df_res[cols_ordem].sort_values(["feature_set", "f1", "auc"], ascending=[True, False, False]).reset_index(drop=True)
    return df_res, grids


def plot_curvas_elbow_silhouette(df_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    # Elbow
    ax[0].plot(df_metrics['k'], df_metrics['inertia'], marker='o')
    ax[0].set_title('Curva do Cotovelo (Inércia por k)')
    ax[0].set_xlabel('k')
    ax[0].set_ylabel('Inércia')
    ax[0].grid(True, alpha=0.3)
    # Silhouette
    ax[1].plot(df_metrics['k'], df_metrics['silhouette'], marker='o')
    ax[1].set_title('Índice de Silhueta por k')
    ax[1].set_xlabel('k')
    ax[1].set_ylabel('Silhouette')
    ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_roc_melhores(df_comp: pd.DataFrame, grids_map: Dict[str, GridSearchCV],
                      X_test_map: Dict[str, pd.DataFrame], y_test: pd.Series, top_n: int = 3) -> None:
    """Plota curvas ROC dos top_n modelos por F1 considerando ambos os feature sets."""
    df_sorted = df_comp.sort_values(['f1', 'auc'], ascending=[False, False]).head(top_n)
    plt.figure(figsize=(7, 5))
    for _, row in df_sorted.iterrows():
        feat = row['feature_set']
        modelo = row['modelo']
        gs = grids_map[(feat, modelo)]
        est = gs.best_estimator_

        if hasattr(est.named_steps['clf'], 'predict_proba'):
            y_proba = est.predict_proba(X_test_map[feat])[:, 1]
        else:
            try:
                score = est.decision_function(X_test_map[feat])
                y_proba = (score - score.min()) / (score.max() - score.min() + 1e-9)
            except Exception:
                y_proba = est.predict(X_test_map[feat]).astype(float)

        RocCurveDisplay.from_predictions(y_test, y_proba, name=f"{feat} | {modelo}")
    plt.title("Curvas ROC — Top modelos (por F1)")
    plt.show()


def avaliar_influencia_k(X_train: pd.DataFrame, y_train: pd.Series,
                         X_test: pd.DataFrame, y_test: pd.Series,
                         k_base: int) -> pd.DataFrame:
    """Avalia a influência do número de clusters no desempenho do SVM RBF
    (feature set expandido). Testa [k_base-2, k_base-1, k_base, k_base+1, k_base+2].
    """
    ks_test = sorted(set([max(2, k_base-2), max(2, k_base-1), k_base, k_base+1, k_base+2]))
    linhas = []
    for k in ks_test:
        Xtr_exp, Xte_exp, _, _ = treinar_kmeans_e_distancia(X_train, X_test, k=k)
        gs, m = treinar_avaliar_grid('svm_rbf', Xtr_exp, y_train, Xte_exp, y_test, cv_splits=5)
        linhas.append({"k": k, **m, "best_params": gs.best_params_})
        print(f"k={k} => F1={m['f1']:.4f} | AUC={m['auc']:.4f}")
    return pd.DataFrame(linhas).sort_values(['f1','auc'], ascending=[False, False]).reset_index(drop=True)


# ========================
# Q1 — Clusterização K-Means
# ========================
print("\\n=======================\\nQ1 — Clusterização K-Means\\n=======================")
df = carregar_dataset_spotify()
X, y = preparar_dados(df)
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2)

# Heurística: avaliar k em 2..15 no treino, com amostra p/ silhouette
ks = list(range(2, 16))
df_k = avaliar_kmeans_treino(X_train, ks=ks, amostra_sil=12000)
display(df_k)

# Visualização Elbow + Silhouette
plot_curvas_elbow_silhouette(df_k)

k_escolhido = escolher_k(df_k)
print(f"k escolhido (Q1): {k_escolhido}")

# =====================================
# Q2 — Criação de Feature de Distância
# =====================================
print("\\n=====================================\\nQ2 — Criação de Feature de Distância\\n=====================================")
X_train_exp, X_test_exp, km_obj, scaler_km = treinar_kmeans_e_distancia(X_train, X_test, k=k_escolhido)

print("Colunas originais:", X_train.columns.tolist()[:8], "... (+", len(X_train.columns), "features)")
print("Colunas expandidas:", X_train_exp.columns.tolist()[-3:], "... total:", len(X_train_exp.columns))

# =====================================================
# Q3 — Modelagem (SVM linear, polinomial, RBF e Random Forest) + GridSearch
# =====================================================
print("\\n=====================================================\\nQ3 — Modelagem + GridSearch (SVMs e Random Forest)\\n=====================================================")

# Conjunto ORIGINAL de features
df_res_orig, grids_orig = comparar_modelos("original", X_train, y_train, X_test, y_test)

# Conjunto EXPANDIDO (com dist_to_centroid)
df_res_exp, grids_exp = comparar_modelos("expandido", X_train_exp, y_train, X_test_exp, y_test)

# Junta resultados
df_comp = pd.concat([df_res_orig, df_res_exp], ignore_index=True)
display(df_comp)

# ==================================
# Q4 — Avaliação (Teste: métricas + ROC)
# ==================================
print("\\n=================================\\nQ4 — Avaliação de Modelos (Teste)\\n=================================")

# Melhor por F1 em cada feature set
melhor_orig = df_res_orig.iloc[0]
melhor_exp  = df_res_exp.iloc[0]
print("Melhor (original):", melhor_orig['modelo'], "| F1:", round(melhor_orig['f1'],4), "| AUC:", round(melhor_orig['auc'],4))
print("Melhor (expandido):", melhor_exp['modelo'],  "| F1:", round(melhor_exp['f1'],4),  "| AUC:", round(melhor_exp['auc'],4))

# Curvas ROC (Top 3 no geral)
grids_map = {}
for _, row in df_comp.iterrows():
    grids_map[(row['feature_set'], row['modelo'])] = (grids_exp if row['feature_set']=='expandido' else grids_orig)[row['modelo']]

X_test_map = {"original": X_test, "expandido": X_test_exp}
plot_roc_melhores(df_comp, grids_map, X_test_map, y_test, top_n=3)

# ======================================================
# Q5 — Análise Comparativa: impacto das features de clusterização
# ======================================================
print("\\n======================================================\\nQ5 — Análise Comparativa (Original x Expandido)\\n======================================================")

# Tabela resumida por modelo (média por feature_set) — aqui só 1 execução, então apenas pivot
tabela_comp = df_comp.pivot_table(index='modelo', columns='feature_set', values=['accuracy','precision','recall','f1','auc'])
display(tabela_comp)

# Visual simples da melhora de F1 por modelo com feature expandida
try:
    modelos = df_res_orig['modelo'].tolist()
    f1_gain = []
    for m in modelos:
        f1_o = float(df_res_orig[df_res_orig['modelo']==m]['f1'].values[0])
        f1_e = float(df_res_exp[df_res_exp['modelo']==m]['f1'].values[0])
        f1_gain.append(f1_e - f1_o)
    plt.figure(figsize=(7,4))
    plt.bar(modelos, f1_gain)
    plt.title("Ganho de F1 ao adicionar 'dist_to_centroid' (Expandido - Original)")
    plt.ylabel("ΔF1")
    plt.xlabel("Modelo")
    plt.grid(True, axis='y', alpha=0.3)
    plt.show()
except Exception as e:
    print("(Aviso) Falha ao plotar ganhos de F1:", e)

# ===================================================================
# Q6 — Influência do número de clusters (variação em torno de k escolhido)
# ===================================================================
print("\\n===================================================================\\nQ6 — Influência do número de clusters (variação em torno de k escolhido)\\n===================================================================")
df_infl_k = avaliar_influencia_k(X_train, y_train, X_test, y_test, k_base=k_escolhido)
display(df_infl_k)

# =====================
# Encerramento / Log
# =====================
print("\\n✔ Execução concluída.\\n- Dataset numérico preparado.\\n- K avaliado (cotovelo + silhouette) e escolhido.\
\\n- Feature 'dist_to_centroid' criada e adicionada em treino/teste.\
\\n- Modelos SVM (linear, polinomial, RBF) e Random Forest com GridSearch (5-fold).\
\\n- Métricas no teste: accuracy, precision, recall, F1 e AUC-ROC.\
\\n- Comparação Original x Expandido e análise da influência do k.")